# FreshIQ — Teaching Notebook### Read this page before running anythingYou are going to build a program that looks at a photo of a fruit and says two things:> *"That's a **banana**, and it's **spoiled**."*By the end you will understand every step, not just have run it. **No machine-learningbackground is assumed.** Every term is explained the first time it appears.---## First, the one idea everything else rests onNormal programming is: *you write the rules.*```pythonif brown_pixels > 30%:    return "spoiled"```That works for simple things and falls apart here. How brown is brown? What about a banana,where brown flecks are *normal*? What about shadows? You'd write a thousand rules and still lose.**Machine learning flips it around: you provide the answers, and the computer works out the rules.**You hand it 71,000 photos, each already labelled "fresh" or "rotten". It looks for whatever visualpatterns separate the two groups. You never tell it *what* to look for. That's the whole trick.The thing it produces — the learned rules — is called a **model**. Training is the process ofproducing one. That's the entire notebook in two sentences.---## What you'll actually do here| Step | In plain English ||---|---|| 1–2 | Find the photos and see what we've got || 3–4 | Make a list of every photo and its correct answer || 5 | **Hunt for duplicate photos** — skip this and your results are fake || 6 | Split the photos into three piles: learn / check / final exam || 7–8 | Set up the photo-feeding machinery || 9 | Build the model || 10–11 | Train it || 12 | Find out honestly how good it is || 13–14 | Shrink it so it runs on a tiny computer || 15 | Package it for the FreshIQ robot |Roughly **2–4 hours** on Kaggle's free GPU.---## Before you start — three clicks1. **Add Data** (right sidebar) → search `food-freshness-dataset` → add ulnnproject's2. **Settings → Accelerator → GPU** — without this, training takes about a day instead of an hour3. Then Run All---## Glossary — bookmark this, come back when a word confuses you| Term | What it means ||---|---|| **Model** | The learned rules. A file full of numbers that turns a photo into an answer. || **Training** | Showing the model photos and correcting it until it stops being wrong. || **Label** | The correct answer for one photo (`fresh` or `spoiled`). || **Class** | One possible answer. We have 2 condition classes and 13 food classes. || **Feature** | A visual pattern the model has learned to notice (an edge, a fuzzy texture, a colour). || **Layer** | One processing stage inside the model. Photos flow through a stack of them. || **Neural network** | A model made of stacked layers. That's all the word means here. || **Weights / parameters** | The adjustable numbers *inside* the model. Training tunes these. || **Epoch** | One complete pass through all the training photos. || **Batch** | A small group of photos processed together (we use 256) — GPUs like bulk. || **Learning rate** | Step size when correcting the model. Too big overshoots; too small crawls. || **Loss** | A score of how wrong the model is. Training = making this number go down. || **Overfitting** | Memorising the training photos instead of learning general rules. || **Transfer learning** | Starting from a model that already learned to see, instead of from scratch. || **Tensor** | A grid of numbers. A colour photo is one: height × width × 3. || **Inference** | Using a finished model to answer a question. The opposite of training. || **Quantisation** | Shrinking a model by storing its numbers less precisely. || **TFLite** | A file format for models that run on small devices like a Raspberry Pi. |

---# 0. The one rule we must not breakThis notebook doesn't produce a model for its own sake. The FreshIQ robot's code will load it,so we have to hand it exactly what it expects.Here's the code in the robot that prepares a photo (`ai/preprocessing.py`):```pythondef prepare_for_model(frame, input_size):    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)   # put colours in R,G,B order    resized = cv2.resize(rgb, (input_size, input_size))    tensor = resized.astype(np.float32) / 255.0     # squash 0-255 down to 0.0-1.0    return np.expand_dims(tensor, axis=0)```**Why divide by 255?** A pixel's brightness is stored as a whole number from 0 (black) to 255(brightest). Neural networks behave much better when their inputs are small numbers near zero, sowe divide everything by 255 to get a range of 0.0 to 1.0. This is called **normalising**.### The rule> **The model must be trained on numbers in the range 0.0–1.0, because that is what it will be> fed when it runs for real.**If you train on 0–255 and then feed it 0.0–1.0, the model still runs. It produces confident-lookingpercentages. They are **complete nonsense**. Nothing crashes, nothing warns you. This is the singlemost common way people ship a broken model, and it's why we keep saying it.*(MobileNet, the model we build on, actually wants −1.0 to +1.0. We handle that with a conversionlayer glued to the front of the model, so the model itself does the final adjustment. Section 9.)*

---# 1. SetupLoading our tools and setting every adjustable number in one place.**About the libraries:**- `numpy` — fast maths on big grids of numbers- `pandas` — spreadsheet-like tables in Python- `PIL` — opening and resizing images- `tensorflow` — the machine-learning library; builds and trains the model- `matplotlib` — charts**About `SEED = 1337`:** lots of steps here involve randomness (shuffling, splitting). A "seed"makes the randomness *repeatable* — run it twice, get identical results. Without it you could nevertell whether a change you made helped, or whether you just got lucky. The number itself is arbitrary.

In [ ]:
import os, json, time, random, warningsfrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutorimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom PIL import Imageimport tensorflow as tfwarnings.filterwarnings("ignore")SEED = 1337random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)# ==================== SETTINGS — everything adjustable lives here ====================# --- PICK ONE. Everything else is derived from it. ---------------------------##   "max"      Best model that still runs comfortably on an Arduino UNO Q.#              6-10 hours on a T4. This is the one to use.#   "balanced" The earlier settings. ~1.5 hours. Fine for a quick check.#   "tiny"     Only for a genuinely tiny board (Pi Zero class, 512 MB, no GPU).PRESET = "max"# WHY "max" IS SAFE ON THE UNO Q# The UNO Q's Linux side is a quad-core Kryo with 2-4 GB of RAM, an Adreno GPU# and a Hexagon DSP. It is not a microcontroller. The old alpha=0.35 / 128px# settings were sized for a 512 MB Pi Zero and left most of the board idle.##                    params   int8 size   rel. compute   UNO Q verdict#   tiny  0.35@128    0.45M     0.65 MB        1x        trivial#   max   1.00@160    2.28M     2.4  MB      ~13x        comfortable## 13x the arithmetic on a board that was doing nothing, for 5x the parameters# to learn with. That is the trade worth making.if PRESET == "max":    IMG_SIZE = 160       # not 224: source photos are 128px native, so 224 is                         # mostly interpolation. 160 gains real detail from the                         # higher-resolution datasets without inventing any.    ALPHA = 1.0          # full-width MobileNetV2 - 2.2M params instead of 0.4M    BATCH = 128          # smaller batch; the bigger model needs more GPU memory    EPOCHS_FROZEN = 30    EPOCHS_FINETUNE = 25    LR_FROZEN = 1e-3    LR_FINETUNE = 5e-5   # gentler still - there is far more here to damage    UNFREEZE_LAYERS = 60 # deeper than the usual 30; more data supports it    USE_COSINE = True    # smooth LR decay rather than step dropselif PRESET == "balanced":    IMG_SIZE, ALPHA, BATCH = 128, 0.35, 256    EPOCHS_FROZEN, EPOCHS_FINETUNE = 15, 8    LR_FROZEN, LR_FINETUNE = 1e-3, 1e-4    UNFREEZE_LAYERS, USE_COSINE = 30, Falseelse:  # tiny    IMG_SIZE, ALPHA, BATCH = 128, 0.35, 256    EPOCHS_FROZEN, EPOCHS_FINETUNE = 15, 8    LR_FROZEN, LR_FINETUNE = 1e-3, 1e-4    UNFREEZE_LAYERS, USE_COSINE = 30, False# Pretrained weights only exist for alpha in {0.35,0.5,0.75,1.0,1.3,1.4}# at input sizes {96,128,160,192,224}. Stray outside and ImageNet init fails.FOOD_LOSS_WEIGHT = 0.5   # food identity matters half as much as condition# --- "That isn't produce" ----------------------------------------------------# Attach any dataset of NON-food photos (people, hands, furniture, animals) and# name it here. Every image in a matching dataset is labelled food="not_food".## WHY THIS MATTERS: without a not_food class the model has no way to say "that# is not produce". Show it your hand and it must pick one of the 13 fruits, and# it will do so confidently. A reject class is what turns a demo into something# you can point at anything.## Negatives train the FOOD head only. They have no freshness, so their condition# loss is masked out (sample weight 0) - see section 8. Nothing is invented.NEGATIVE_KEYWORDS = ["natural-images", "human-faces", "intel-image",                     "caltech", "objects", "not-food", "negatives"]MAX_NEGATIVE_FRACTION = 0.15   # cap negatives at 15% of the produce photos, so                               # they cannot swamp the classes you actually care about# --- Data cleaning -----------------------------------------------------------RUN_AUDIT = True         # ~5 min. Finds duplicate and greyscale photos. DO NOT SKIP.HAMMING_MAX = 4          # how similar two photos must be to count as duplicatesDROP_GREYSCALE = True    # throw away colourless photos (explained in section 5)SPLIT = (0.70, 0.15, 0.15)   # learn / check / final-exam proportionsMAX_WEIGHT, MIN_WEIGHT = 5.0, 0.2   # imbalance correction limits (section 8)# ====================================================================================WORK = Path("/kaggle/working")OUT = WORK / "models"; OUT.mkdir(parents=True, exist_ok=True)CACHE_DIR = WORK / "tfcache"; CACHE_DIR.mkdir(parents=True, exist_ok=True)print("TensorFlow version:", tf.__version__)gpus = tf.config.list_physical_devices("GPU")if gpus:    print("GPU found:", gpus[0].name)    for g in gpus:        tf.config.experimental.set_memory_growth(g, True)else:    print("!!! NO GPU !!!  Settings -> Accelerator -> GPU, then restart the notebook.")

### What just happenedNothing was trained. We imported libraries and defined constants.**If you saw "NO GPU"** — stop and fix it. A GPU (graphics processor) does thousands of smallcalculations at once, which is exactly what training needs. On the regular processor this notebookwould run for roughly a day.**About mixed precision (next cell):** computers store decimals with a certain number of digits.Fewer digits = faster maths but less precision. Some GPUs (T4) have special hardware forhalf-precision and get a big speedup; others (P100) don't. We check which one Kaggle gave us andonly switch it on when it helps.

In [ ]:
USE_MIXED_PRECISION = Falseif gpus:    name = tf.config.experimental.get_device_details(gpus[0]).get("device_name", "")    print("GPU model:", name)    if any(k in name.upper() for k in ("T4", "V100", "A100", "L4")):        from tensorflow.keras import mixed_precision        mixed_precision.set_global_policy("mixed_float16")        USE_MIXED_PRECISION = True        print("-> half precision ON (this GPU has hardware for it)")    else:        print("-> full precision (half precision wouldn't help on this GPU)")

---# 2. Find the photosKaggle puts datasets in `/kaggle/input/`. That folder is **read-only** — you can look but nottouch. Anything you create goes in `/kaggle/working/`.We *search* for the right folder rather than typing a path, because whoever uploaded the datasetchose the folder names, and hard-coding a path you haven't verified is how notebooks break.We're looking for a folder containing both something like "Fresh" and something like "Rotten".

In [ ]:
INPUT_ROOT = Path("/kaggle/input")def dataset_roots(base):    # Kaggle mounts either as  /kaggle/input/<slug>/    # or, once you attach several, /kaggle/input/datasets/<owner>/<name>/ .    # Treating /kaggle/input/* as the roots collapses every dataset into a single    # entry called "datasets", which makes the per-dataset counts below useless    # and hides a dataset that contributed nothing.    roots = []    if not base.exists():        return roots    for p in sorted(base.iterdir()):        if not p.is_dir():            continue        if p.name == "datasets":            for owner in sorted(x for x in p.iterdir() if x.is_dir()):                for ds in sorted(x for x in owner.iterdir() if x.is_dir()):                    roots.append(ds)        else:            roots.append(p)    return rootsattached = dataset_roots(INPUT_ROOT)print(f"{len(attached)} dataset(s) attached:")for p in attached:    print("   ", p.relative_to(INPUT_ROOT))if not attached:    print("   NONE - use 'Add Data' in the right sidebar")# --- Every food name we know, and every way people spell it -------------------# Different datasets use different words for the same vegetable, and several# contain outright typos. Everything is folded to one canonical name here.FOOD_ALIASES = {    "apple": "apple", "apples": "apple",    "banana": "banana", "bananas": "banana",    "bellpepper": "bell_pepper", "bellpeppers": "bell_pepper", "pepper": "bell_pepper",    "capsicum": "capsicum", "capciscum": "capsicum",    "bittergourd": "bitter_gourd", "bittergroud": "bitter_gourd", "bittergourds": "bitter_gourd",    "carrot": "carrot", "carrots": "carrot",    "cucumber": "cucumber", "cucumbers": "cucumber",    "mango": "mango", "mangos": "mango", "mangoes": "mango",    "okra": "okra", "okara": "okra", "okras": "okra", "ladyfinger": "okra",    "orange": "orange", "oranges": "orange",    "potato": "potato", "potatoes": "potato",    "strawberry": "strawberry", "strawberries": "strawberry",    "tomato": "tomato", "tomatoes": "tomato",    # foods the extra datasets bring that the first one did not have    "grape": "grape", "grapes": "grape",    "guava": "guava", "guavas": "guava",    "jujube": "jujube", "pomegranate": "pomegranate",    "lemon": "lemon", "lime": "lime",    "onion": "onion", "onions": "onion",    "peach": "peach", "pear": "pear", "plum": "plum", "kiwi": "kiwi",    "cherry": "cherry", "cherries": "cherry", "watermelon": "watermelon",    "cabbage": "cabbage", "cauliflower": "cauliflower", "brinjal": "eggplant",    "eggplant": "eggplant", "aubergine": "eggplant",    # more spellings the other datasets use    "patato": "potato", "potatos": "potato",    "tamto": "tomato", "tomatoe": "tomato", "tamato": "tomato",    "aple": "apple", "bannana": "banana", "orage": "orange",    "cucumbers": "cucumber", "carrott": "carrot",    "chilli": "chilli", "chili": "chilli", "greenchilli": "chilli",    "raddish": "radish", "radish": "radish",    "beetroot": "beetroot", "cucmber": "cucumber",    "papaya": "papaya", "pineapple": "pineapple", "lime": "lime",    "soyabeans": "soybean", "soybean": "soybean",}# Longest first, so "bellpepper" wins over "pepper" and "bittergourd" over nothing.ALIAS_ORDER = sorted(FOOD_ALIASES, key=len, reverse=True)# NOTE the order: "unhealthy" contains "healthy", so spoilage words are tested first.SPOILED_WORDS = ("rotten", "rotton", "stale", "spoil", "unhealthy", "unfresh", "bad", "damaged")FRESH_WORDS = ("fresh", "healthy", "good", "ripe")def _clean(s):    s = s.lower()    for ch in " -_()[].,":        s = s.replace(ch, "")    return sdef read_path(rel_parts):    # Infer (condition, food) from the folder path. This is what lets ONE scanner    # cope with every dataset's different layout:    #   Fresh/FreshApple/x.jpg     Apple__Healthy/x.jpg     train/rottenbanana/x.jpg    #    # DEEPEST COMPONENT WINS, and that matters enormously. One recommended dataset    # sits in a folder called "Fruit and Vegetable Diseases (Healthy vs Rotten)" -    # a name containing BOTH words. Searching the whole path as one string would    # mark every photo in it spoiled. Reading innermost-first, the real class    # folder ("Apple__Healthy") answers before the container ever gets a say.    parts = [_clean(p) for p in rel_parts]    condition = food = None    for p in reversed(parts):        if condition is None:            for w in SPOILED_WORDS:                if w in p:                    condition = "spoiled"; break            if condition is None:                for w in FRESH_WORDS:                    if w in p:                        condition = "fresh"; break        if food is None:            for alias in ALIAS_ORDER:                if alias in p:                    food = FOOD_ALIASES[alias]; break        if condition and food:            break    return condition, food

Now we walk every attached dataset and collect every image. This takes a minute or two — it istouching tens of thousands of directory entries.Anything whose condition or food we cannot work out from its path gets **counted and reported**rather than silently dropped. If that number is large, a dataset uses vocabulary we have nottaught the scanner, and you can add it to `FOOD_ALIASES` above.

In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}def is_negative_dataset(name):    n = name.lower()    return any(k in n for k in NEGATIVE_KEYWORDS)rows, skipped, neg_rows = [], [], []print(f"{'dataset':<40}{'used':>10}{'skipped':>10}  kind")print("-" * 74)for ds in attached:    n_before, skip_before = len(rows), sum(n for _, n in skipped)    negative = is_negative_dataset(ds.name)    n_neg_before = len(neg_rows)    for dirpath, dirnames, filenames in os.walk(ds):        imgs = [f for f in filenames if os.path.splitext(f)[1].lower() in EXTS]        if not imgs:            continue        rel = Path(dirpath).relative_to(ds).parts        if negative:            # Everything here is "not produce". Skip any folder that clearly IS            # produce (natural-images ships a 'fruit' class, for example) - it            # would teach the model that fruit is not food.            _c, f_guess = read_path(rel)            if f_guess is not None:                skipped.append((str(Path(dirpath).relative_to(INPUT_ROOT)), len(imgs)))                continue            for f in imgs:                neg_rows.append((os.path.join(dirpath, f), "__none__", "not_food", ds.name))            continue        condition, food = read_path(rel)        if condition is None or food is None:            skipped.append((str(Path(dirpath).relative_to(INPUT_ROOT)), len(imgs)))            continue        for f in imgs:            rows.append((os.path.join(dirpath, f), condition, food, ds.name))    used = (len(neg_rows) - n_neg_before) if negative else (len(rows) - n_before)    skip = sum(n for _, n in skipped) - skip_before    kind = "NEGATIVES (not_food)" if negative else "produce"    flag = "  <- CONTRIBUTED NOTHING" if used == 0 else ""    print(f"{ds.name[:38]:<40}{used:>10,}{skip:>10,}  {kind}{flag}")# Cap the negatives so they cannot dominate the batch.if neg_rows:    cap = int(MAX_NEGATIVE_FRACTION * max(len(rows), 1))    if len(neg_rows) > cap:        rng0 = np.random.default_rng(SEED)        keep = rng0.choice(len(neg_rows), size=cap, replace=False)        print(f"\ncapping negatives: {len(neg_rows):,} -> {cap:,} "              f"({MAX_NEGATIVE_FRACTION:.0%} of produce)")        neg_rows = [neg_rows[i] for i in keep]    rows.extend(neg_rows)    print(f"added {len(neg_rows):,} not_food photos")df = pd.DataFrame(rows, columns=["path", "condition", "food", "source"])print("-" * 64)print(f"TOTAL: {len(df):,} photos | {df.food.nunique()} foods | "      f"{df.source.nunique()} dataset(s) actually used")if skipped:    tot = sum(n for _, n in skipped)    print(f"\n{tot:,} photos in {len(skipped)} folders could NOT be labelled from their path.")    print("If that number is large, a dataset uses words the scanner does not know -")    print("add them to FOOD_ALIASES / SPOILED_WORDS / FRESH_WORDS above. Largest:")    for p, n in sorted(skipped, key=lambda x: -x[1])[:12]:        print(f"   {n:>7,}  {p}")

### The counts, per dataset and per food

In [ ]:
if df.source.nunique() > 1:    print("Photos per dataset:")    print(df.groupby(["source", "condition"]).size().unstack(fill_value=0).to_string(), "\n")table = df.pivot_table(index="food", columns="condition", values="path",                       aggfunc="count", fill_value=0)table["total"] = table.sum(axis=1)display(table.sort_values("total", ascending=False))print(f"\nMost photos: {table.total.max():,}   Fewest: {table.total.min():,}"      f"   Ratio: {table.total.max()/max(table.total.min(),1):.0f}x")

### What just happenedEvery attached dataset was walked, and each photo's **condition** and **food** were read from itsfolder path. That is why one scanner copes with all of them: `Fresh/FreshApple/`,`Apple__Healthy/` and `train/rottenbanana/` all resolve to the same two labels.**Look at the ratio.** If one food has forty times more photos than another, that is**class imbalance** — a genuine problem, and section 8 deals with it.**Notice what is missing:** no `Ripe` anywhere. These datasets only know fresh and rotten. That iswhy FreshIQ reports two classes — a class with no training data could never be predicted, soprinting it would only mislead.> **Adding more datasets is safe.** Several of these are rebuilds of each other, so they overlap> heavily. Section 5 fingerprints every photo and groups the duplicates, so an overlap becomes a> deduplicated group rather than a leak. Attach as many as you like.

### What just happened — and one thing to decideThat ratio at the bottom is your imbalance. If it says `40x`, one food has forty times more photosthan another.**`bell_pepper` and `capsicum` are the same vegetable.** This dataset was stitched together fromthree separate sources, and they used different names. Two folders for one vegetable means themodel gets asked to tell apart things that are identical — an impossible task it will fail at,muddying the results.Look at the sample photos in the next section and decide. If they look like the same thing, comeback and set `MERGE_PEPPERS = True`.*(Sometimes "capsicum" means specifically the green ones and "bell pepper" the red/yellow ones, inwhich case keeping them separate is reasonable. Your eyes decide, not me.)*

In [ ]:
MERGE_PEPPERS = False    # <-- flip to True after looking at section 4if MERGE_PEPPERS and "capsicum" in set(df.food):    df.loc[df.food == "capsicum", "food"] = "bell_pepper"    print(f"Merged. {df.food.nunique()} foods remain.")FOODS = sorted(df.food.unique())CONDITIONS = ["fresh", "spoiled"]    # ORDER MATTERS - written into labels.json laterprint("Conditions:", CONDITIONS)print("Foods:", FOODS)

> **Why "order matters":** the finished model outputs a bare list of numbers like `[0.13, 0.87]`.> Nothing in that list says which number is which. The **position** is the only clue —> position 0 is `fresh`, position 1 is `spoiled`, because that's the order we chose here. We save> this order into `labels.json` so the robot reads them the same way round. Swap it and every> answer inverts.

---# 4. Look at the photosThe most skipped step in machine learning, and the one that saves the most time.You cannot debug a model you haven't looked at the inputs for. Five minutes here beats five hourswondering why the numbers are strange.**What to look for:**- Do the "fresh" photos actually look fresh?- Are the labels believable, or are some obviously wrong?- Are backgrounds wildly different between the two groups? *(If every rotten photo has a dark  background, the model will learn "dark background = rotten" and be useless in the real world.  This is called a **shortcut**, and models find them constantly.)*

In [ ]:
def show_grid(frame, title, n=8):    if len(frame) == 0:        print(f"(no photos for: {title})"); return    sample = frame.sample(min(n, len(frame)), random_state=SEED)    fig, axes = plt.subplots(1, len(sample), figsize=(2 * len(sample), 2.5))    axes = np.atleast_1d(axes)    for ax, (_, r) in zip(axes, sample.iterrows()):        ax.imshow(Image.open(r.path).convert("RGB")); ax.axis("off")        ax.set_title(f"{r.food}\n{r.condition}", fontsize=8)    fig.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()show_grid(df[df.condition == "fresh"],   "FRESH — should look edible")show_grid(df[df.condition == "spoiled"], "SPOILED — should look rotten")for pep in ("bell_pepper", "capsicum"):    show_grid(df[df.food == pep], f"'{pep}' — same vegetable as the other?", n=6)

---# 5. The most important section: finding duplicate photos## The problem, with an analogyImagine studying for an exam with 100 practice questions, then sitting an exam made of **those exactsame 100 questions**. You'd score 100%. Would that prove you understood the subject? No — it provesyou memorised the answers.That is exactly what happens if the same photo appears in both your training pile and yourtesting pile. The model memorises it, aces the test, and you conclude it's brilliant. Then it meetsa real banana and falls over.## Why this dataset specifically has the problemRead the dataset's own description:> *"refined through preprocessing and **augmentation**... grayscale conversion, edge detection, and> **brightness adjustment**"***Augmentation** means making extra copies of photos with small changes — brighter, rotated,flipped. It's a legitimate technique. But it means **near-identical copies of the same originalphoto are already scattered through these folders**, and if we split randomly, twins land onopposite sides.## How we catch themWe can't compare files byte-for-byte — a brightened copy has completely different bytes. We need tocompare what the photos *look like*.**A perceptual hash (dHash)** does this in four steps:1. Shrink the photo to a tiny 9×8 grid — fine detail vanishes, overall structure survives2. Convert to greyscale — colour differences stop mattering3. For each pixel, ask: *is it brighter than the pixel to its right?* → yes/no4. That's 64 yes/no answers. Write them as 64 bits. **That's the fingerprint.**Brightening a photo raises *every* pixel equally, so "is this brighter than its neighbour" gives the**same answers**. The fingerprint barely moves. That's the whole idea.Two photos whose fingerprints differ in ≤ 4 of 64 bits are almost certainly the same original.**One pass, three jobs.** While we're opening every file anyway, we also record how colourful eachphoto is and its dimensions. Reading 71,000 files is the slow part; doing three things per read isnearly free.

In [ ]:
def audit_one(path, hash_size=8):    # Returns (fingerprint, colourfulness 0-255, width, height), or None if unreadable.    try:        im = Image.open(path)        im.draft("L", (hash_size + 1, hash_size))   # fast partial JPEG decode        w, h = im.size        rgb = im.convert("RGB")        # Colourfulness: for each pixel, how far apart are its most and least        # intense channels? Grey pixels have R=G=B, so this is ~0 for greyscale.        small = np.asarray(rgb.resize((16, 16)), dtype=np.int16)        mx, mn = small.max(axis=2), small.min(axis=2)        sat = float(np.mean(np.where(mx > 0, (mx - mn) * 255.0 / np.maximum(mx, 1), 0)))        # dHash: 9x8 greyscale, compare each pixel to its right neighbour -> 64 bits        g = np.asarray(rgb.convert("L").resize((hash_size + 1, hash_size)), dtype=np.int16)        bits = (g[:, 1:] > g[:, :-1]).flatten()        v = 0        for b in bits:            v = (v << 1) | int(b)        return v, sat, w, h    except Exception:        return Noneif RUN_AUDIT:    t0 = time.time()    # Threads help here because most of the time is spent waiting on disk, not computing.    with ThreadPoolExecutor(max_workers=16) as ex:        results = list(ex.map(audit_one, df.path.tolist()))    print(f"Audited {len(results):,} photos in {time.time()-t0:.0f} seconds")    df["dhash"] = [r[0] if r else None for r in results]    df["sat"]   = [r[1] if r else np.nan for r in results]    df["w"]     = [r[2] if r else np.nan for r in results]    df["h"]     = [r[3] if r else np.nan for r in results]    broken = df.dhash.isna().sum()    if broken:        print(f"Dropping {broken} unreadable files")        df = df[df.dhash.notna()].reset_index(drop=True)    print("\nPhoto sizes found:")    print(df.groupby(["w", "h"]).size().sort_values(ascending=False).head(5))else:    df["dhash"], df["sat"] = range(len(df)), 99.0

### Sub-step: colourless photosOur model's whole job is spotting **discolouration** — brown patches, fuzzy grey mould, dullskin. A greyscale photo has thrown that information away. An edge-detected photo (white outlines onblack) has thrown away even more.Training on those teaches the model to judge rot from shape and brightness alone — which willnot transfer to your robot's colour camera.In the histogram below, a healthy colour dataset shows a broad hump well right of zero. **A spikeat the far left is the greyscale copies.**

In [ ]:
if RUN_AUDIT:    plt.figure(figsize=(10, 3))    plt.hist(df.sat.dropna(), bins=80, color="#4a9")    plt.axvline(12, color="crimson", ls="--", lw=2, label="cutoff: below this = colourless")    plt.xlabel("colourfulness (0 = pure grey)"); plt.ylabel("number of photos")    plt.legend(); plt.title("How colourful is each photo?"); plt.show()    grey = df[df.sat < 12]    print(f"{len(grey):,} photos ({100*len(grey)/len(df):.1f}%) look colourless")    if len(grey):        show_grid(grey, "Suspected greyscale / edge-detected photos", n=8)        if DROP_GREYSCALE:            before = df.groupby("food").size()            # NEVER apply this to the negatives.            #            # The colour filter exists because colour IS the freshness signal.            # But a not_food photo has no freshness - it only has to teach the            # model "this is not produce", and a muted photo of a building does            # that perfectly well.            #            # Real-world scenes (streets, glaciers, faces, buildings) are far            # less saturated than studio produce shots, so this cutoff wiped out            # 99.7% of the negatives on the first attempt - 25,926 became 71 -            # and the reject class silently disappeared from the model.            keep = (df.sat >= 12) | (df.food == "not_food")            df = df[keep].reset_index(drop=True)            after = df.groupby("food").size()            print(f"Removed. {len(df):,} colour photos remain.\n")            print(f"{'food':<16}{'before':>9}{'after':>9}{'kept':>8}")            for f in before.index:                a = int(after.get(f, 0))                print(f"{f:<16}{before[f]:>9,}{a:>9,}{100*a/before[f]:>7.0f}%")

### Drop the foods that did not surviveLook at that `kept` column. Some foods in this dataset were supplied almost entirely as greyscaleconversions, so removing colourless photos wipes them out — one may drop from ten thousand photos tosingle digits.A class with three surviving photos cannot be learned, and leaving it in does active harm: it dragsthe food accuracy around and fills the confusion matrix with zero-support rows that read as failureswhen they are really just empty.So we drop any food below a minimum and **print exactly which ones**. Silently keeping them makesthe results look worse than they are; silently dropping them overstates your coverage.**The condition head is unaffected.** It never cared which fruit it was looking at, and it still hasevery surviving photo to learn from.

In [ ]:
MIN_PHOTOS_PER_FOOD = 300      # below this there is not enough left to learn fromcounts = df.groupby("food").size().sort_values()# not_food is never pruned. It is the reject class, not a food, and losing it# silently removes the model's ability to say "that is not produce".too_few = counts[(counts < MIN_PHOTOS_PER_FOOD) & (counts.index != "not_food")]if len(too_few):    print("DROPPED from the food head - too few colour photos survived:")    for f, n in too_few.items():        print(f"   {f:<16} {n:>6,}")    df = df[~df.food.isin(too_few.index)].reset_index(drop=True)    print()else:    print("Every food cleared the minimum.\n")print(f"{len(df):,} photos across {df.food.nunique()} foods:")for f, n in df.groupby("food").size().sort_values(ascending=False).items():    print(f"   {f:<16} {n:>7,}")# ---- REBUILD THE CLASS LIST ------------------------------------------------# FOODS was first computed back in section 3, BEFORE any filtering. If we leave# it alone, every food we just dropped still gets an output neuron that can# never fire - dead capacity, and rows of zeros in the confusion matrix that# look like failures but are simply empty. Recompute it here, after the cut._before = len(FOODS)FOODS = sorted(df.food.unique())if len(FOODS) != _before:    print(f"\nFood head resized: {_before} classes -> {len(FOODS)}")print(f"Food head will predict: {FOODS}")print("\nThe condition head (fresh/spoiled) is unaffected - it never cared which "      "fruit it was looking at, and still uses every one of these photos.")

### Sub-step: grouping the duplicatesComparing all 71,000 photos to each other would be 2.5 *billion* comparisons. Too slow.**Blocking** fixes it: near-identical fingerprints usually share a chunk of bits, so we bucketphotos by 16-bit chunks and only compare within buckets. Comparisons drop by thousands of times.Then **union-find** merges the pairs into groups. It's a small algorithm for answering"are these two in the same group?" quickly. If A matches B, and B matches C, all three end up inone group even though we never compared A to C directly.Every photo ends up with a `group_id`. **All photos sharing a group_id came from one originalphoto, and must never be split across piles.**

In [ ]:
class UnionFind:    # Each item starts in its own group; union() merges two groups.    def __init__(self, n): self.parent = list(range(n))    def find(self, x):        while self.parent[x] != x:            self.parent[x] = self.parent[self.parent[x]]   # path compression            x = self.parent[x]        return x    def union(self, a, b):        ra, rb = self.find(a), self.find(b)        if ra != rb:            self.parent[rb] = raif RUN_AUDIT:    t0 = time.time()    hashes = df.dhash.astype("int64").to_numpy()    uf = UnionFind(len(df))    for shift in (0, 16, 32, 48):          # four different 16-bit windows        buckets = {}        for i, hv in enumerate(hashes):            buckets.setdefault((int(hv) >> shift) & 0xFFFF, []).append(i)        for idxs in buckets.values():            if len(idxs) < 2 or len(idxs) > 4000:    # skip useless mega-buckets                continue            for a in range(len(idxs)):                ha = int(hashes[idxs[a]])                for b in range(a + 1, len(idxs)):                    # Count differing bits between the two fingerprints.                    if bin(ha ^ int(hashes[idxs[b]])).count("1") <= HAMMING_MAX:                        uf.union(idxs[a], idxs[b])    df["group_id"] = [uf.find(i) for i in range(len(df))]    n_groups = df.group_id.nunique()    copies = len(df) / n_groups    print(f"Grouping took {time.time()-t0:.0f}s")    print(f"\n{len(df):,} photos  ->  {n_groups:,} genuinely different originals")    print(f"Average {copies:.2f} copies of each original")    print(f"Biggest cluster: {df.groupby('group_id').size().max()} copies of one photo")    if copies > 1.5:        print(f"\n>>> A random split would have leaked roughly "              f"{100*(1 - 1/copies):.0f}% of test photos into training. <<<")else:    df["group_id"] = range(len(df))

### What just happenedIf it said *"average 4 copies of each original"*, then a naive random split would have put about**3 of every 4 test photos' twins into the training pile**. The model would have scored superblyand learned very little.You now have a `group_id` on every photo. The next section uses it.> This one section is the difference between a number you can defend and a number that just> looks good. Most tutorials skip it entirely.

---# 6. Three pilesStandard practice, and each pile has a distinct job:| Pile | Share | Purpose | Does the model learn from it? ||---|---|---|---|| **train** | 70% | The photos it learns from | Yes || **validation** | 15% | Progress checks during training | No — but it decides when to stop || **test** | 15% | The final exam | No — touched **once**, at the very end |**Why not just two piles?** Because we use validation to make decisions — when to stop, whichlearning rate, which threshold. Anything you make decisions from, you've partly tuned toward. So wekeep a third pile that influences *nothing*, and only look at it once. That's the honest number.> **The rule:** if you look at the test score and then change something and re-run, that test score> is dead. It became a validation score the moment it influenced you.**Stratified** means each pile gets a representative mix of every food and condition — not all thebananas in one pile. We do this by splitting *within* each (condition, food) group.And critically: **we shuffle and split `group_id`s, never individual photos.** That's what keepsduplicates from crossing over.

In [ ]:
rng = np.random.RandomState(SEED)# Collapse to one row per group, tagged with that group's condition and food.groups = (df.groupby("group_id")            .agg(condition=("condition", lambda s: s.mode()[0]),                 food=("food", lambda s: s.mode()[0]),                 n=("path", "size"))            .reset_index())assignment = {}for (cond, food), cell in groups.groupby(["condition", "food"]):    ids = cell.group_id.to_numpy().copy()    rng.shuffle(ids)    n_train = int(round(SPLIT[0] * len(ids)))    n_val   = int(round(SPLIT[1] * len(ids)))    for gid in ids[:n_train]:                    assignment[gid] = "train"    for gid in ids[n_train:n_train + n_val]:     assignment[gid] = "validation"    for gid in ids[n_train + n_val:]:            assignment[gid] = "test"df["split"] = df.group_id.map(assignment)print(df.split.value_counts().to_string())print("\nCondition mix in each pile (should be near-identical):")print(pd.crosstab(df.split, df.condition, normalize="index").round(3).to_string())# Prove it rather than trust it.max_splits = df.groupby("group_id").split.nunique().max()assert max_splits == 1, f"LEAK! A group spans {max_splits} piles"print("\nVERIFIED: no duplicate group spans two piles.")train_df = df[df.split == "train"].reset_index(drop=True)val_df   = df[df.split == "validation"].reset_index(drop=True)test_df  = df[df.split == "test"].reset_index(drop=True)

> **On that `assert`:** it stops the notebook dead if the guarantee is violated. Checking your own> assumptions in code — rather than believing the logic is right — is a habit worth forming. It costs> one line and catches the errors you'd otherwise never notice.

---# 7. The photo-feeding machineThe GPU can process photos far faster than the disk can supply them. Feed it naively and it sitsidle most of the time. So we build a **pipeline**:```read file → decode JPEG → resize → CACHE → shuffle → augment → scale to 0-1 → batch → prefetch                                     ▲                  ▲                            cache BEFORE augment   augment AFTER cache```**Three decisions worth understanding:****1. Cache as whole numbers, not decimals.** 71,000 photos at 128×128×3 is about **3.5 GB** storedas bytes (0–255), but **14 GB** as decimals. One fits on Kaggle's disk; the other doesn't. So wecache the raw bytes and convert to decimals afterwards, every time.**2. Cache *before* augmenting — this one is subtle and important.** Augmentation applies randomchanges. If you cache after it, you freeze one particular set of random changes and replay theidentical "random" photos every epoch. The augmentation silently stops doing anything at all.**3. Prefetch.** While the GPU works on batch 5, the CPU prepares batch 6. Neither waits.---### What augmentation is, and why ours is deliberately timidAugmentation shows the model slightly altered copies — flipped, rotated a little, slightly brighter— so it learns *"a banana is a banana regardless of angle"* rather than memorising exact pixels.Many tutorials go hard on this. **We must not.** Look at what we're detecting: brown patches, darkspots, discolouration. Crank the colour distortion up and you will *manufacture a brown patch on afresh banana* — while still labelling it fresh. You'd be actively teaching the model that brownpatches mean fresh.So: horizontal flips, small rotations, small zoom, gentle brightness and contrast. Nothing thatinvents damage. *(The first epoch is slow while the cache fills. Later epochs are much faster.)*

In [ ]:
n_cond, n_food = len(CONDITIONS), len(FOODS)cond_idx = {c: i for i, c in enumerate(CONDITIONS)}food_idx = {f: i for i, f in enumerate(FOODS)}augment = tf.keras.Sequential([    tf.keras.layers.RandomFlip("horizontal"),          # mirror left-right    tf.keras.layers.RandomRotation(0.08),              # up to about +/- 29 degrees...    tf.keras.layers.RandomZoom(0.10),                  # in or out by up to 10%    tf.keras.layers.RandomBrightness(0.12, value_range=(0.0, 1.0)),    tf.keras.layers.RandomContrast(0.12),], name="augmentation")def decode(path):    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="area")    return tf.cast(img, tf.uint8)          # keep as bytes so the cache stays smallIS_NEG = "__none__"          # marker for a not_food row: no freshness existsdef condition_target(c):    # One-hot for a real condition; ALL ZEROS for a not_food photo.    v = [0.0] * n_cond    if c != IS_NEG:        v[cond_idx[c]] = 1.0    return v# cond_w / food_w are PER-HEAD sample weights.## They have to be separate because a not_food photo has no freshness. Its# condition target is all zeros and its condition weight is 0, so it contributes# nothing to the condition loss while still teaching the food head what "not# produce" looks like. Sharing one weight array across both heads would either# train freshness on meaningless targets or throw the negative away entirely.def make_dataset(frame, training, cache_name, cond_w=None, food_w=None):    paths = frame.path.to_numpy()    yc = tf.constant([condition_target(c) for c in frame.condition], tf.float32)    yf = tf.one_hot([food_idx[f] for f in frame.food], n_food)    if cond_w is None:        ds = tf.data.Dataset.from_tensor_slices((paths, yc, yf))        ds = ds.map(lambda p, c, f: (decode(p), c, f), num_parallel_calls=tf.data.AUTOTUNE)    else:        ds = tf.data.Dataset.from_tensor_slices(            (paths, yc, yf, cond_w.astype("float32"), food_w.astype("float32")))        ds = ds.map(lambda p, c, f, wc, wf: (decode(p), c, f, wc, wf),                    num_parallel_calls=tf.data.AUTOTUNE)    ds = ds.cache(str(CACHE_DIR / cache_name))     # <- raw bytes, before augmentation    if training:        ds = ds.shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)    def finish(img, c, f, wc=None, wf=None):        x = tf.cast(img, tf.float32) / 255.0        # THE CONTRACT: 0-255 -> 0.0-1.0        if training:            x = augment(x, training=True)        y = {"condition": c, "food": f}        if wc is None:            return (x, y)        return (x, y, {"condition": wc, "food": wf})    ds = ds.map(finish, num_parallel_calls=tf.data.AUTOTUNE)    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)print("Pipeline defined. Datasets get built in the next section.")

---# 8. Fixing the imbalance## The problemSuppose 90% of your photos are apples. The model discovers it can score 90% by answering "apple"every single time — without learning anything. It's not cheating; it's doing exactly what youasked, because you asked for accuracy.## The fix: sample weightsWe tell the model that **mistakes on rare groups cost more**. A photo from a group with a tenth ofthe average count counts about ten times as much when the model is being corrected.The formula per (condition, food) group:$$\text{weight} = \frac{\text{total photos}}{\text{number of groups} \times \text{photos in this group}}$$*(The usual approach is `class_weight` in Keras, but that doesn't work for models with two outputs.Sample weights — one number per photo — do.)*## Why we cap the weights, and what that costsThe smallest group might have 300 photos against 13,700 in the largest. Uncapped, that's a **45×**weight. Now one mislabelled photo in that tiny group pushes the model as hard as 45 correct ones.Training becomes unstable — the loss bounces around and never settles.Capping to `[0.2, 5.0]` says: *fix most of the imbalance, but don't let 300 photos run the show.***This is a real trade-off, not a trick.** You accept some leftover bias toward the common classesin exchange for training that actually converges. Being able to name what you gave up is thedifference between engineering and cargo-culting.

In [ ]:
cell_counts = train_df.groupby(["condition", "food"]).size()n_cells, n_total = len(cell_counts), len(train_df)raw     = {k: n_total / (n_cells * v) for k, v in cell_counts.items()}clipped = {k: float(np.clip(v, MIN_WEIGHT, MAX_WEIGHT)) for k, v in raw.items()}print(f"{'group':<30}{'photos':>9}{'raw wt':>9}{'capped':>9}")print("-" * 57)for k in sorted(raw, key=lambda k: cell_counts[k]):     # rarest first    flag = "  <- capped" if abs(raw[k] - clipped[k]) > 1e-9 else ""    print(f"{k[0]+'/'+k[1]:<30}{cell_counts[k]:>9,}{raw[k]:>9.2f}{clipped[k]:>9.2f}{flag}")base_w = train_df.set_index(["condition", "food"]).index.map(clipped).to_numpy().astype("float32")# Food head: every photo counts, including the not_food ones.food_w = base_w.copy()# Condition head: not_food photos are MASKED OUT. They have no freshness, so# letting them contribute would train the model on a meaningless target.cond_w = base_w.copy()is_neg = (train_df.condition == IS_NEG).to_numpy()cond_w[is_neg] = 0.0print(f"\ncondition head trains on {(~is_neg).sum():,} photos "      f"({is_neg.sum():,} not_food masked out)")print(f"food head trains on      {len(train_df):,} photos")train_ds = make_dataset(train_df, True, "train.cache", cond_w=cond_w, food_w=food_w)val_ds   = make_dataset(val_df,   False, "val.cache")test_ds  = make_dataset(test_df,  False, "test.cache")print(f"\ntrain {len(train_df):,}   validation {len(val_df):,}   test {len(test_df):,}")

---# 9. Building the model## Transfer learning — why we don't start from nothingTraining a vision model from scratch means it starts knowing *nothing*: not edges, not colour, nottexture. It would need millions of photos and enormous compute to get there.**MobileNetV2** has already been trained on 1.4 million photos of everything from dogs toumbrellas. In the process it learned to detect edges, colour gradients, textures, fuzzy surfaces,smooth surfaces. Those are exactly the building blocks "mould looks fuzzy and desaturated" ismade of.So we **keep its eyes and replace its opinions.** We take the whole visual system, throw away thefinal "is it a dog?" layer, and bolt on our own small layer that asks "is it rotten?" instead.That's transfer learning, and it's why 71,000 photos is enough.## Our architecture```        photo (128 x 128 x 3), numbers 0.0 - 1.0                      │              Rescaling(2, -1)          converts 0..1 to -1..+1                      │                 (MobileNet was trained expecting that)         MobileNetV2 (alpha = 0.35)     the borrowed eyes                      │                 output: a 4x4 grid, 400+ values deep        GlobalAveragePooling2D          average each depth-slice -> one flat list                      │              Dropout(0.3)              anti-memorisation (below)                    /   \        condition (2)   food (13)       our two new layers```**GlobalAveragePooling2D** — MobileNet outputs a small grid of numbers per feature. We just averageeach one down to a single value. It's a way of asking *"how much of this feature is anywhere in thephoto"* while forgetting exactly where, which is what we want: mould is mould wherever it sits.**Dropout(0.3)** — during training, 30% of the values are randomly zeroed each step. It soundsdestructive; it works. The model can't lean on any single feature (it might vanish), so it's forcedto spread its reasoning across many. That's a direct defence against memorisation.## Why two heads instead of two separate modelsBoth questions need the same eyes. Sharing one body means:- **One file to ship.** On a 512 MB Pi, running two models means two copies of everything resident.- **One pass** answers both questions.- **They help each other.** Learning "this is a banana" gives the shared body a better sense of  shape and texture, which makes the rot question easier too.The cost: the two heads compete for the same limited capacity. `FOOD_LOSS_WEIGHT = 0.5` tells themodel condition is twice as important, because condition is the actual product.**One last detail** — the final layers are forced to full precision (`dtype="float32"`). Under halfprecision the softmax overflows and produces `nan` (not-a-number), which poisons everythingsilently. Standard fix; easy to forget.

In [ ]:
def build_model(fine_tune=False):    base = tf.keras.applications.MobileNetV2(        input_shape=(IMG_SIZE, IMG_SIZE, 3),        alpha=ALPHA,        include_top=False,        # drop the original "is it a dog?" layer        weights="imagenet",       # KEEP everything it learned from 1.4M photos    )    base.trainable = fine_tune    # stage 1: frozen. stage 2: partly unfrozen.    if fine_tune:        for layer in base.layers[:-UNFREEZE_LAYERS]:            layer.trainable = False        # BatchNorm is the exception: its running mean/variance must stay frozen        # during fine-tuning. Letting them update on small batches destabilises        # everything downstream, and it is the classic reason a fine-tune run        # trains fine and then evaluates far worse than it should.        for layer in base.layers:            if isinstance(layer, tf.keras.layers.BatchNormalization):                layer.trainable = False    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")    x = tf.keras.layers.Rescaling(2.0, offset=-1.0)(inputs)   # 0..1  ->  -1..+1    x = base(x, training=False)    x = tf.keras.layers.GlobalAveragePooling2D()(x)    x = tf.keras.layers.Dropout(0.3)(x)    condition = tf.keras.layers.Dense(n_cond, activation="softmax",                                      dtype="float32", name="condition")(x)    food = tf.keras.layers.Dense(n_food, activation="softmax",                                 dtype="float32", name="food")(x)    return tf.keras.Model(inputs, {"condition": condition, "food": food}, name="freshiq")def compile_model(m, lr):    m.compile(        optimizer=tf.keras.optimizers.Adam(lr),        # "categorical_crossentropy" = the standard how-wrong-are-we score for        # pick-one-of-N problems. Low when confident and right; high when        # confident and wrong.        loss={"condition": "categorical_crossentropy",              "food": "categorical_crossentropy"},        loss_weights={"condition": 1.0, "food": FOOD_LOSS_WEIGHT},        metrics={"condition": ["accuracy"], "food": ["accuracy"]},    )    return mmodel = compile_model(build_model(fine_tune=False), LR_FROZEN)total = model.count_params()trainable = sum(int(tf.size(w)) for w in model.trainable_weights)print(f"Numbers inside the model : {total:,}")print(f"Numbers we'll adjust now : {trainable:,}  ({100*trainable/total:.1f}%)  <- just the heads")print(f"Size as decimals         : ~{total*4/1e6:.1f} MB")print(f"Size after shrinking     : ~{total/1e6:.1f} MB   <- what goes on the Pi")

---# 10. Stage 1 — train the new heads onlyMobileNet's body is **frozen**: its numbers can't change. Only our two new layers learn.**Why freeze it?** Our new layers start with random numbers, so at first they're wildly wrong. Wildlywrong produces huge corrections. If those flowed back into MobileNet's carefully-trained body, they'dscramble the very features we borrowed it for — before those features had a chance to be useful.So: let the heads get roughly right first, *then* consider touching the body. That's stage 2.## Reading the outputEach epoch prints something like:```loss: 0.31 - condition_accuracy: 0.89 - val_loss: 0.35 - val_condition_accuracy: 0.87```- `loss` / `accuracy` — on the **training** photos- `val_loss` / `val_accuracy` — on the **validation** photos it has never learned from**`val_loss` is the number that matters.** Training loss almost always improves; that just meansit's memorising. Validation loss improving means it's genuinely learning.## The four safety nets| Callback | What it prevents ||---|---|| **EarlyStopping** (patience 5) | Overfitting. Stops when `val_loss` hasn't improved for 5 epochs, and **rewinds to the best epoch**. || **ReduceLROnPlateau** | Getting stuck. Halves the step size when progress stalls — a stride that was right at the start is too clumsy near the target. || **ModelCheckpoint** | Losing your best model to a bad final epoch or a session timeout. || **CSVLogger** | Not being able to diagnose it afterwards. |

In [ ]:
keras_path = OUT / "freshiq_model.keras"callbacks = [    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,                                     restore_best_weights=True, verbose=1),    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,                                         patience=3, min_lr=1e-6, verbose=1),    tf.keras.callbacks.ModelCheckpoint(str(keras_path), monitor="val_condition_accuracy",                                       mode="max", save_best_only=True),    tf.keras.callbacks.CSVLogger(str(OUT / "training_log.csv")),]print("Stage 1 starting. The FIRST epoch is slow (filling the cache) - this is normal.\n")t0 = time.time()hist1 = model.fit(train_ds, validation_data=val_ds,                  epochs=EPOCHS_FROZEN, callbacks=callbacks, verbose=1)print(f"\nStage 1 done in {(time.time()-t0)/60:.1f} minutes")

---# 11. Stage 2 — fine-tuningNow we unfreeze **the top 30 layers** of MobileNet and continue at a **ten times smaller**learning rate.**Why only the top?** Layers form a hierarchy. Early ones detect universal things — edges, colourblobs — that are identical whether you're looking at dogs or bananas. Those need no change. Latelayers hold high-level, dataset-specific concepts. ImageNet's "this is a dog's snout" is useless tous, but that same capacity retrained becomes "this is a mould patch."**Why the tiny learning rate?** These weights are already good. At the stage-1 rate we'd take hugesteps and destroy them within a few batches — ending up worse than we started. This is the singlemost common fine-tuning mistake.

In [ ]:
ft_model = build_model(fine_tune=True)ft_model.set_weights(model.get_weights())     # carry over everything learned in stage 1compile_model(ft_model, LR_FINETUNE)trainable = sum(int(tf.size(w)) for w in ft_model.trainable_weights)print(f"Now adjusting {trainable:,} numbers "      f"({100*trainable/ft_model.count_params():.1f}%) - heads plus the top of the body\n")ft_callbacks = [    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,                                     restore_best_weights=True, verbose=1),    tf.keras.callbacks.ModelCheckpoint(str(keras_path), monitor="val_condition_accuracy",                                       mode="max", save_best_only=True),    tf.keras.callbacks.CSVLogger(str(OUT / "training_log_finetune.csv")),]if USE_COSINE:    # Cosine decay: the learning rate eases smoothly to zero over the run instead    # of dropping in steps. Late in training you want very small steps to settle    # into a minimum rather than bouncing around it, and cosine gets there    # gradually rather than waiting for a plateau to be detected.    def cosine(epoch):        import math        return LR_FINETUNE * 0.5 * (1 + math.cos(math.pi * epoch / EPOCHS_FINETUNE))    ft_callbacks.append(tf.keras.callbacks.LearningRateScheduler(cosine, verbose=0))    print(f"cosine LR: {LR_FINETUNE:.1e} -> ~0 over {EPOCHS_FINETUNE} epochs")else:    ft_callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1))t0 = time.time()hist2 = ft_model.fit(train_ds, validation_data=val_ds,                     epochs=EPOCHS_FINETUNE, callbacks=ft_callbacks, verbose=1)print(f"\nStage 2 done in {(time.time()-t0)/60:.1f} minutes")model = ft_model

In [ ]:
def plot_curves(hists, names):    fig, axes = plt.subplots(1, 3, figsize=(16, 4))    offset = 0    for h, nm in zip(hists, names):        e = range(offset, offset + len(h.history["loss"])); offset += len(h.history["loss"])        axes[0].plot(e, h.history["loss"], label=f"{nm} train")        axes[0].plot(e, h.history["val_loss"], "--", label=f"{nm} validation")        axes[1].plot(e, h.history["condition_accuracy"], label=f"{nm} train")        axes[1].plot(e, h.history["val_condition_accuracy"], "--", label=f"{nm} validation")        axes[2].plot(e, h.history["food_accuracy"], label=f"{nm} train")        axes[2].plot(e, h.history["val_food_accuracy"], "--", label=f"{nm} validation")    for ax, t in zip(axes, ["loss (lower is better)", "condition accuracy", "food accuracy"]):        ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=8); ax.grid(alpha=.3)    plt.tight_layout(); plt.show()plot_curves([hist1, hist2], ["stage 1", "stage 2"])

### How to read those curves**Healthy:** solid and dashed lines fall together, with a modest gap between them.**Overfitting:** the solid (training) line keeps dropping while the dashed (validation) linebottoms out and turns **upward**. The gap widens. That's the model memorising instead of learning.A little overfitting at the tail is fine — EarlyStopping already rewound to the best epoch. A *large*gap means you need more data, stronger augmentation, or fewer epochs.

---# 12. The honest verdictThe test pile has been untouched. We look **once**.## Why accuracy is a bad headline numberSay 95% of photos are fresh. A model that answers "fresh" every time scores **95% accuracy** and iscompletely worthless. Accuracy hides which mistakes you're making.## The confusion matrixA table of what was *actually* true against what the model *said*:|  | said fresh | said spoiled ||---|---|---|| **really fresh** |  correct |  wasteful || **really spoiled** |  **dangerous** |  correct |The diagonal is right; everything else is wrong. And the two wrong cells are **not equally bad**:- **Really spoiled → said fresh** — you passed rotten food. **This is the expensive error.**- **Really fresh → said spoiled** — you binned a good tomato. Wasteful. Nobody gets ill.So the number to report for a food-inspection machine is **recall on spoiled**: of all the genuinelyspoiled items, what fraction did we catch?

In [ ]:
scores = model.evaluate(test_ds, return_dict=True, verbose=0)print("TEST RESULTS (first and only look)")for k, v in scores.items():    print(f"  {k:<30} {v:.4f}")pred = model.predict(test_ds, verbose=1)# Food head is scored on EVERYTHING, including the not_food photos - correctly# rejecting a photo of a person is part of the job.y_pred_f = pred["food"].argmax(1)y_true_f = np.array([food_idx[f] for f in test_df.food])# Condition head is scored ONLY on real produce. A not_food photo has no# freshness, so including it would be scoring an answer that does not exist.real = (test_df.condition != IS_NEG).to_numpy()y_pred_c = pred["condition"].argmax(1)[real]y_true_c = np.array([cond_idx[c] for c in test_df.condition[real]])print(f"condition scored on {real.sum():,} produce photos "      f"({(~real).sum():,} not_food excluded)")

In [ ]:
def confusion(y_true, y_pred, names, title, figsize=(7, 6)):    k = len(names)    cm = np.zeros((k, k), dtype=int)    for t, p in zip(y_true, y_pred):        cm[t, p] += 1    print(f"\n=== {title} ===")    print(f"{'class':<16}{'precision':>10}{'recall':>9}{'f1':>8}{'photos':>9}")    for i, n in enumerate(names):        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp        pr = tp / (tp + fp) if tp + fp else 0.0     # of what it CALLED this, how much was?        rc = tp / (tp + fn) if tp + fn else 0.0     # of what really IS this, how much did it find?        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0   # balance of the two        print(f"{n:<16}{pr:>10.3f}{rc:>9.3f}{f1:>8.3f}{cm[i].sum():>9,}")    print(f"{'OVERALL':<16}{'':>10}{'':>9}{cm.trace()/max(cm.sum(),1):>8.4f}")    norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)    fig, ax = plt.subplots(figsize=figsize)    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)    ax.set_xticks(range(k)); ax.set_xticklabels(names, rotation=45, ha="right")    ax.set_yticks(range(k)); ax.set_yticklabels(names)    ax.set_xlabel("what the model SAID"); ax.set_ylabel("what was TRUE"); ax.set_title(title)    for i in range(k):        for j in range(k):            if cm[i, j]:                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,                        color="white" if norm[i, j] > 0.5 else "black")    plt.tight_layout(); plt.show()    return cmcm_c = confusion(y_true_c, y_pred_c, CONDITIONS, "CONDITION", (4.5, 4))cm_f = confusion(y_true_f, y_pred_f, FOODS, "FOOD IDENTITY", (9, 8))

**Precision vs recall, in one line each:**- **Precision** — of everything it *called* spoiled, how much really was? (low = crying wolf)- **Recall** — of everything that really *was* spoiled, how much did it catch? (low = missing rot)For us, **recall on spoiled is the one that matters.**

In [ ]:
i_sp, i_fr = cond_idx["spoiled"], cond_idx["fresh"]n_spoiled = cm_c[i_sp].sum()missed = cm_c[i_sp, i_fr]recall_spoiled = cm_c[i_sp, i_sp] / max(n_spoiled, 1)print("=" * 62)print(f"  SPOILED RECALL: {recall_spoiled:.4f}    <-- report THIS number")print("=" * 62)print(f"  spoiled, called fresh : {missed:,} of {n_spoiled:,} "      f"({100*missed/max(n_spoiled,1):.2f}%)   <- rotten food passed as good")print(f"  fresh, called spoiled : {cm_c[i_fr, i_sp]:,} of {cm_c[i_fr].sum():,}"      f"   <- good food binned (just wasteful)")confidence = pred["condition"].max(1)[real]print(f"\n  average confidence : {confidence.mean():.3f}")print(f"  unsure (<60%)      : {(confidence < 0.6).sum():,} photos")real_df = test_df[real].reset_index(drop=True)show_grid(real_df.iloc[np.argsort(confidence)[:8]],          "Least confident answers - check these for WRONG LABELS in the dataset", n=8)# --- Does it actually know what is NOT produce? --------------------------if "not_food" in FOODS:    ni = food_idx["not_food"]    neg = ~real    if neg.sum():        caught = (y_pred_f[neg] == ni).mean()        print(f"\n  NOT-PRODUCE RECALL : {caught:.4f}  "              f"({int(caught*neg.sum()):,} of {neg.sum():,} non-food photos rejected)")    # And the error that would embarrass you: real produce called not_food.    false_reject = (y_pred_f[real] == ni).mean()    print(f"  produce wrongly called not_food: {false_reject:.4f}")

> **Why look at the unsure photos?** When a model is uncertain, it's often because the photo is> genuinely ambiguous — or **mislabelled**. Finding a handful of wrong labels here tells you> something real about your data's quality, and about the ceiling on any model trained from it.### Choosing where to draw the lineBy default we take whichever class scores higher. But we've established the two errors aren'tequally costly — so why treat 50/50 as the cutoff?Lowering the bar for calling something spoiled catches more rot at the price of more false alarms.For food inspection that's usually the right trade.**Crucially, we pick the threshold using validation, not test.** Picking it from test would beexactly the tuning-on-the-final-exam mistake we've been avoiding all notebook.

In [ ]:
val_real = (val_df.condition != IS_NEG).to_numpy()val_scores = model.predict(val_ds, verbose=0)["condition"][val_real, i_sp]val_truth = np.array([cond_idx[c] for c in val_df.condition[val_real]]) == i_spprint(f"{'threshold':>10}{'spoiled caught':>16}{'fresh kept':>13}{'accuracy':>11}")print("-" * 50)for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:    said_spoiled = val_scores >= t    caught = (said_spoiled & val_truth).sum() / max(val_truth.sum(), 1)    kept   = (~said_spoiled & ~val_truth).sum() / max((~val_truth).sum(), 1)    print(f"{t:>10.2f}{caught:>16.4f}{kept:>13.4f}{(said_spoiled == val_truth).mean():>11.4f}")SPOILED_THRESHOLD = 0.50    # <-- pick from the table aboveprint(f"\nUsing {SPOILED_THRESHOLD}. Lower catches more rot but bins more good food.")

---# 13. Shrinking it for the PiThree files come out of this, and FreshIQ needs all three:| File | Why ||---|---|| `freshiq_model.keras` | The full-quality original. **Keep it** — you can't un-shrink the small one. || `freshiq_model.tflite` | The shrunk version that ships to the board || `labels.json` | Which output number means what. **Without it the model is gibberish.** |## What quantisation actually isRight now every number in the model is stored as a 32-bit decimal — very precise, 4 bytes each.**Quantisation** stores them as 8-bit whole numbers (0–255) instead. One byte each.- **4× smaller file**- **Much faster on ARM chips**, which handle whole-number maths far better than decimalsThe catch: to squeeze decimals into 0–255 sensibly, you need to know the actual range of valuesthat flow through the model — and that depends on real photos. That's what a **representativedataset** is for: we push a few hundred genuine training photos through so the converter canmeasure the real ranges and pick sensible conversions.**Skip that step and you get a model that's fast, small, and wrong.**

In [ ]:
model.save(keras_path)print(f"Saved {keras_path.name}  ({keras_path.stat().st_size/1e6:.2f} MB)\n")labels = {    "type": "condition+food",           # tells ai/model.py which backend to build    "classes": CONDITIONS,              # ORDER of the condition output    "foods": FOODS,                     # ORDER of the food output    "input_size": IMG_SIZE,    "architecture": f"MobileNetV2-alpha{ALPHA}",    "preprocessing": "RGB float32 in [0,1]; model applies Rescaling(2,-1) internally",    "spoiled_threshold": SPOILED_THRESHOLD,    # Tells inference that this model can reject non-produce. When the food head    # predicts "not_food", the freshness verdict must be suppressed rather than    # reported - it is meaningless for a photo of a hand.    "not_food_class": "not_food" if "not_food" in FOODS else None,    "test_metrics": {        "condition_accuracy": float(cm_c.trace() / max(cm_c.sum(), 1)),        "spoiled_recall": float(recall_spoiled),        "food_accuracy": float(cm_f.trace() / max(cm_f.sum(), 1)),        "test_photos": int(len(test_df)),    },    "notes": "Trained on Kaggle ulnnproject/food-freshness-dataset (Apache 2.0). "             "Web photos of supermarket produce, NOT photos from the FreshIQ rig. "             "Fine-tune on real rig photos before trusting these numbers in production.",}with open(OUT / "labels.json", "w") as fh:    json.dump(labels, fh, indent=2)print(json.dumps(labels, indent=2))

In [ ]:
rep_paths = train_df.sample(min(300, len(train_df)), random_state=SEED).path.tolist()def representative_data():    # Same preprocessing as real inference - otherwise the calibration is wrong.    for p in rep_paths:        im = Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))        yield [np.expand_dims(np.asarray(im, dtype=np.float32) / 255.0, 0)]# ---- Undo mixed precision before converting --------------------------------# Mixed precision leaves float16 operations all through the graph. Full int8# conversion cannot legalise those, and the conversion dies with##     error: 'tf.Mul' op is neither a custom op nor a flex op#     error: 'tf.Conv2D' op is neither a custom op nor a flex op   ... etc## once per layer - hundreds of lines of it. Rebuilding the same architecture# under a float32 policy and copying the trained weights across gives a clean# graph to quantise. The weights are identical; only the storage type differs.if USE_MIXED_PRECISION:    from tensorflow.keras import mixed_precision    mixed_precision.set_global_policy("float32")    export_model = build_model(fine_tune=True)    export_model.set_weights(model.get_weights())    export_model.save(keras_path)          # re-save as a clean float32 model    print("Rebuilt under float32 for export (mixed precision was on).\n")else:    export_model = modelconv = tf.lite.TFLiteConverter.from_keras_model(export_model)conv.optimizations = [tf.lite.Optimize.DEFAULT]conv.representative_dataset = representative_dataconv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]tfl = conv.convert()tflite_path = OUT / "freshiq_model.tflite"tflite_path.write_bytes(tfl)print(f"Shrunk model : {tflite_path.stat().st_size/1e6:.2f} MB")# A full-precision TFLite too, in case shrinking cost too much accuracy.conv_f = tf.lite.TFLiteConverter.from_keras_model(export_model)conv_f.optimizations = [tf.lite.Optimize.DEFAULT](OUT / "freshiq_model_float.tflite").write_bytes(conv_f.convert())print(f"Backup (full precision): {(OUT/'freshiq_model_float.tflite').stat().st_size/1e6:.2f} MB")

### Which output is which?Our model has two outputs. **TFLite does not guarantee they come out in the order we declaredthem.** If they swap, food probabilities get read as condition probabilities and every answer isgarbage — with nothing crashing to tell you.We work out which is which by **width** (one has 2 values, the other 13 — unambiguous) and recordthe answer in `labels.json` so the robot never has to guess.

In [ ]:
interp = tf.lite.Interpreter(model_content=tfl)interp.allocate_tensors()inp, outs = interp.get_input_details()[0], interp.get_output_details()print("input :", inp["shape"], inp["dtype"].__name__)for i, o in enumerate(outs):    print(f"output[{i}]: {o['shape']}  {o['dtype'].__name__}")i_cond_out = next(i for i, o in enumerate(outs) if o["shape"][-1] == n_cond)i_food_out = next(i for i, o in enumerate(outs) if o["shape"][-1] == n_food)print(f"\ncondition = output[{i_cond_out}]   food = output[{i_food_out}]")labels["tflite_output_index"] = {"condition": int(i_cond_out), "food": int(i_food_out)}with open(OUT / "labels.json", "w") as fh:    json.dump(labels, fh, indent=2)print("Recorded in labels.json")

In [ ]:
# Re-test using the SHRUNK model itself. Never assume quantisation was free.N = min(3000, len(test_df))sub = test_df.sample(N, random_state=SEED)ok_c = ok_f = sp_total = sp_hit = 0for _, r in sub.iterrows():    im = Image.open(r.path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))    interp.set_tensor(inp["index"], np.expand_dims(np.asarray(im, np.float32) / 255.0, 0))    interp.invoke()    pc = CONDITIONS[int(np.argmax(interp.get_tensor(outs[i_cond_out]["index"])))]    pf = FOODS[int(np.argmax(interp.get_tensor(outs[i_food_out]["index"])))]    ok_c += (pc == r.condition); ok_f += (pf == r.food)    if r.condition == "spoiled":        sp_total += 1; sp_hit += (pc == "spoiled")print(f"Shrunk model on {N:,} test photos")print(f"  condition accuracy : {ok_c/N:.4f}   (before shrinking: {cm_c.trace()/max(cm_c.sum(),1):.4f})")print(f"  food accuracy      : {ok_f/N:.4f}   (before shrinking: {cm_f.trace()/max(cm_f.sum(),1):.4f})")print(f"  spoiled recall     : {sp_hit/max(sp_total,1):.4f}   (before shrinking: {recall_spoiled:.4f})")print("\nDropped more than 1-2 points? Ship freshiq_model_float.tflite instead.")

---# 14. Will it run on a Pi Zero?We time it here using **one processor core**, which is the closest Kaggle can get to a single ARM core.**Be honest about what this measures.** Kaggle's Intel chip is not a Raspberry Pi. Treat the numberbelow as a floor and a sanity check, never a prediction:| Board | Roughly, vs one Kaggle core | Verdict ||---|---|---|| **Pi Zero 2 W** (4 cores, has SIMD) | 4–8× slower | The realistic target. Fine. || **Pi Zero / W** (1 core, ARM11, **no SIMD**) | 20–50× slower | Painful. No official TFLite build exists for it either. || **Pi 4 / 5** | 2–3× slower | Very comfortable. |**The only number that counts is one you measure on the actual board.** Copy the `.tflite` acrossand time it there.

In [ ]:
import statisticssolo = tf.lite.Interpreter(model_content=tfl, num_threads=1)solo.allocate_tensors()si = solo.get_input_details()[0]x = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)for _ in range(10):                       # warm up - first runs are always slower    solo.set_tensor(si["index"], x); solo.invoke()times = []for _ in range(50):    t = time.perf_counter()    solo.set_tensor(si["index"], x); solo.invoke()    times.append((time.perf_counter() - t) * 1000)med = statistics.median(times)print(f"One core, one photo: {med:.1f} ms (median), {sorted(times)[47]:.1f} ms (worst 5%)")print(f"Model size: {len(tfl)/1e6:.2f} MB\n")print("Rough extrapolation - VERIFY ON THE REAL BOARD:")print(f"  Pi Zero 2 W (4 threads) : ~{med*6/3:.0f}-{med*8/3:.0f} ms per photo")print(f"  Pi Zero W (1 core)      : ~{med*20:.0f}-{med*50:.0f} ms per photo")print(f"\nThe tray takes 4 photos per item, so a Pi Zero 2 W spends about "      f"{med*8/3*4/1000:.1f}s per item thinking -")print("far less than the time the tray spends rotating and settling. Comfortable.")

### The other limit: memoryA Pi Zero has **512 MB of RAM, total**, shared with the operating system. Rough budget:- TFLite + this model: **10–20 MB**- NumPy: ~30 MB- **OpenCV: 100+ MB** — and FreshIQ needs it for the quality check and background removalIt fits, but not alongside a desktop. Run **Pi OS Lite** (no graphical desktop). If memory getstight, `opencv-python-headless` drops the GUI parts and a lot of the weight.**And never install full TensorFlow on the Pi** — it's ~400 MB and simply won't fit. Install`tflite-runtime` instead. FreshIQ's `ai/model.py` already prefers it and only falls back to fullTensorFlow, so the code is already right for this.

---# 15. Package it upEverything lands in `/kaggle/working/models/`. Grab `freshiq_models.zip` from the **Output** panelon the right.**Good news on setup:** the FreshIQ repo has already been updated to match this notebook — twoclasses, 128px, α=0.35, all 13 foods. So there's **no config file to copy and no code to patch**.Drop the model files in and it works.

In [ ]:
import shutilzip_path = shutil.make_archive(str(WORK / "freshiq_models"), "zip", str(OUT))print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.2f} MB)\n")for f in sorted(OUT.iterdir()):    print(f"  {f.name:<34} {f.stat().st_size/1e3:>10,.1f} KB")# Save the exact split so the whole thing is reproducible and auditable.df[["path", "condition", "food", "group_id", "split"]].to_csv(    OUT / "split_manifest.csv", index=False)print(f"\nsplit_manifest.csv saved - the exact split, reproducible from SEED={SEED}")

---# Done. What now?### Installing it1. Download `freshiq_models.zip` from the **Output** panel2. Unzip `freshiq_model.tflite`, `freshiq_model.keras` and `labels.json` into `FreshIQcode/models/`3. Test it:   ```   python scripts/test_image.py sample_data/tomato_spoiled.jpg   ```   The `backend` field should now say **`tflite_multihead`** instead of `heuristic`.4. `pytest -q` should still pass.### What you learned- **Transfer learning** — borrow a model that already learned to see- **Two-stage training** — heads first frozen, then gently unfreeze the top- **Data leakage** — and why perceptual hashing is the defence- **Class imbalance** — sample weights, and why capping them is a trade-off you should be able to name- **Overfitting** — what it looks like on a curve, and the callbacks that stop it- **Confusion matrices** — why accuracy is a bad headline- **Quantisation** — 4× smaller, and why it needs real photos to calibrate### The part that decides whether any of this worksThis model learned from **web photos of supermarket produce**. Your robot photographs food on aspecific tray under a specific light. That gap is real, and it is the biggest threat to yournumbers.Once the rig runs, shoot **200–300 photos per class on the actual hardware**, arrange them as:```dataset/train/fresh/tomato/*.jpgdataset/train/spoiled/tomato/*.jpgdataset/validation/...dataset/test/...```then run one more gentle pass:```python training/train.py --data-dir dataset --fine-tune --lr 1e-5 --epochs 10```That single step will improve real-world accuracy more than any amount of architecture tweaking here.**And report the numbers your test set actually supports. Nothing more.**